# PyDI Data Integration Workflow: Products

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with product datasets to showcase the data integration pipeline from information extraction over schema and entity matching to data fusion.

The four product datasets are derived from the **WDC Products benchmark** (Web Data Commons): the same hardware products re-exported under four different attribute-naming conventions. Per-dataset provenance (including the WDC origin) is attached to each DataFrame's `attrs["provenance"]` in Part 1.

## Table of Contents

* [Part 1: Manual Schema Matching](#Part-1:-Manual-Schema-Matching)
    * [Step 1: Schema Matching](#Step-1:-Schema-Matching)
    * [Step 2: Data profiling (again)](#Step-2:-Data-profiling-(again))
* [Part 2: Entity Matching](#Part-2:-Entity-Matching)
    * [Step 1: Blocking](#Step-1:-Blocking)
    * [Step 2: Evaluate Blocking Against Ground Truth](#Step-2:-Evaluate-Blocking-Against-Ground-Truth)
    * [Step 3: Entity Matching with Comparators](#Step-3:-Entity-Matching-with-Comparators)
    * [Step 4: Rule Based Matcher](#Step-4:-Rule-Based-Matcher)
    * [Step 5: Evaluate Matching Against Ground Truth](#Step-5:-Evaluate-Matching-Against-Ground-Truth)
    * [Step 6: 1:1 Refinement (Greedy vs Maximum Bipartite Matching)](#Step-6:-1:1-Refinement-(Greedy-vs-Maximum-Bipartite-Matching))
    * [Step 7: ML-Based Matcher](#Step-7:-ML-Based-Matcher)
* [Part 3: Data Fusion](#Part-3:-Data-Fusion)
    * [Step 1: Define Fusion Strategy](#Step-1:-Define-Fusion-Strategy)
    * [Step 2: Run Fusion](#Step-2:-Run-Fusion)
    * [Step 3: Evaluate Data Fusion](#step-3-evaluate-data-fusion)

## Part 1: Manual Schema Matching

### Step 1: Schema Matching 

In [11]:
import json
import pandas as pd
import logging
import os
import re
from PyDI.io import load_json
from PyDI.io import load_csv
from pathlib import Path

from PyDI.schemamatching import SchemaTranslator
from PyDI.utils import DataProfiler
from PyDI.normalization import load_normalization_spec
from PyDI.entitymatching import RuleBasedMatcher
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import EntityMatchingEvaluator
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import EntityMatchingEvaluator
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm
from PyDI.entitymatching import MaximumBipartiteMatching

from PyDI.entitymatching import MLBasedMatcher, FeatureExtractor
from sklearn.ensemble import RandomForestClassifier

from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, prefer_higher_trust, voting, maximum, minimum
from PyDI.fusion import DataFusionEngine
from PyDI.fusion import exact_match, numeric_tolerance_match
from PyDI.fusion import DataFusionEvaluator, DataFusionStrategy, exact_match, numeric_tolerance_match, tokenized_match


from dotenv import load_dotenv
load_dotenv()


# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
SCHEMA_DIR = INPUT_DIR / "schemamatching"



products_1_source = load_json(INPUT_DIR / "data" / "dataset_1.json",
                              name="products_1")
products_2_source = load_json(INPUT_DIR / "data" / "dataset_2.json",
                              name="products_2")
products_3_source = load_json(INPUT_DIR / "data" / "dataset_3.json",
                              name="products_3")
products_4_source = load_json(INPUT_DIR / "data" / "dataset_4.json",
                              name="products_4")

source_datasets = [products_1_source, products_2_source, products_3_source, products_4_source]
names = ["products_1", "products_2", "products_3", "products_4"]

# Load target schema
with open(SCHEMA_DIR / "products_target_schema.json") as f:
    target_schema = json.load(f)

spec = load_normalization_spec(SCHEMA_DIR / "products_target_schema.json")
target_columns = list(spec.columns.keys())

df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

SCHEMA_MATCHES = {
    "products_1": {  # snake_case catalog feed
        "id": "id",
        "manufacturer": "brand",
        "product_name": "title",
        "product_description": "description",
        "list_price": "price",
        "currency_code": "priceCurrency",
        "cluster_id": "cluster_id",
        "product_url": "url",
        "name_and_description": "title_description",
        "model_name": "model",
        "manufacturer_part_number": "model_number",
        "category": "product_type",
        "gpu_chipset": "chipset_name",
        "video_memory_gb": "vram_gb",
        "capacity_gb": "storage_gb",
        "sequential_read_mb_s": "read_speed_mb_s",
        "sequential_write_mb_s": "write_speed_mb_s",
        "bus_standard": "bus_type",
        "interface": "interface_type",
        "width_millimeters": "width_mm",
        "length_millimeters": "length_mm",
        "height_millimeters": "height_mm",
        "weight_grams": "weight_g",
        "connector": "storage_connection_type",
        "memory_technology": "memory_type",
        "colour": "color",
        "form_factor": "form_factor",
    },
    "products_2": {  # camelCase web API
        "id": "id",
        "brandName": "brand",
        "name": "title",
        "descriptionText": "description",
        "priceAmount": "price",
        "currency": "priceCurrency",
        "cluster_id": "cluster_id",
        "productUrl": "url",
        "titleAndDescription": "title_description",
        "modelName": "model",
        "mpn": "model_number",
        "productCategory": "product_type",
        "chipset": "chipset_name",
        "vramGb": "vram_gb",
        "capacityGb": "storage_gb",
        "readSpeedMbps": "read_speed_mb_s",
        "writeSpeedMbps": "write_speed_mb_s",
        "busType": "bus_type",
        "interfaceType": "interface_type",
        "widthMm": "width_mm",
        "depthMm": "length_mm",
        "heightMm": "height_mm",
        "weightG": "weight_g",
        "connectionType": "storage_connection_type",
        "memoryType": "memory_type",
        "color": "color",
        "formFactor": "form_factor",
    },
    "products_3": {  # PascalCase price list
        "id": "id",
        "Brand": "brand",
        "ProductTitle": "title",
        "Details": "description",
        "Price": "price",
        "Currency": "priceCurrency",
        "cluster_id": "cluster_id",
        "Link": "url",
        "TitleDetails": "title_description",
        "Model": "model",
        "PartNo": "model_number",
        "Type": "product_type",
        "Chipset": "chipset_name",
        "MemorySizeGB": "vram_gb",
        "CapacityGB": "storage_gb",
        "ReadMBs": "read_speed_mb_s",
        "WriteMBs": "write_speed_mb_s",
        "Bus": "bus_type",
        "Interface": "interface_type",
        "WidthMM": "width_mm",
        "LengthMM": "length_mm",
        "HeightMM": "height_mm",
        "WeightG": "weight_g",
        "Connector": "storage_connection_type",
        "MemoryType": "memory_type",
        "Colour": "color",
        "FormFactor": "form_factor",
    },
    "products_4": {  # terse ERP codes
        "id": "id",
        "mfr": "brand",
        "name": "title",
        "desc": "description",
        "amt": "price",
        "cur": "priceCurrency",
        "cluster_id": "cluster_id",
        "link": "url",
        "name_desc": "title_description",
        "mdl": "model",
        "pn": "model_number",
        "cat": "product_type",
        "chip": "chipset_name",
        "vram": "vram_gb",
        "cap_gb": "storage_gb",
        "rd_mbs": "read_speed_mb_s",
        "wr_mbs": "write_speed_mb_s",
        "bus": "bus_type",
        "iface": "interface_type",
        "w_mm": "width_mm",
        "l_mm": "length_mm",
        "h_mm": "height_mm",
        "wt_g": "weight_g",
        "conn": "storage_connection_type",
        "mem": "memory_type",
        "clr": "color",
        "ff": "form_factor",
    },
}


def build_mapping(source_name, target_name, column_map):
    """Build a PyDI SchemaMapping (DataFrame) from a hand-written column map.

    The returned frame has the columns SchemaTranslator expects:
    ``source_dataset``, ``source_column``, ``target_dataset``,
    ``target_column`` and ``score``.
    """
    rows = [
        {
            "source_dataset": source_name,
            "source_column": src,
            "target_dataset": target_name,
            "target_column": tgt,
            "score": 1.0,
        }
        for src, tgt in column_map.items()
    ]
    return pd.DataFrame(
        rows,
        columns=["source_dataset", "source_column", "target_dataset", "target_column", "score"],
    )


translator = SchemaTranslator()

In [12]:
# Build the hand-written mapping for each source and translate it into the
# shared target schema. 
mapping_1 = build_mapping("products_1", "target_schema", SCHEMA_MATCHES["products_1"])
df_final_1 = translator.translate(products_1_source, mapping=mapping_1, normalize=None)

mapping_2 = build_mapping("products_2", "target_schema", SCHEMA_MATCHES["products_2"])
df_final_2 = translator.translate(products_2_source, mapping=mapping_2, normalize=None)

mapping_3 = build_mapping("products_3", "target_schema", SCHEMA_MATCHES["products_3"])
df_final_3 = translator.translate(products_3_source, mapping=mapping_3, normalize=None)

mapping_4 = build_mapping("products_4", "target_schema", SCHEMA_MATCHES["products_4"])
df_final_4 = translator.translate(products_4_source, mapping=mapping_4, normalize=None)

mappings = {
    "products_1": mapping_1,
    "products_2": mapping_2,
    "products_3": mapping_3,
    "products_4": mapping_4,
}

# Downstream entity matching and fusion expect the shared target schema, so the
# translated frames become the working datasets from here on.
products_1_cleaned = df_final_1
products_2_cleaned = df_final_2
products_3_cleaned = df_final_3
products_4_cleaned = df_final_4

datasets = [products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned]

# Sanity check + provenance: every source now exposes exactly the target
# columns, and each frame still carries its WDC benchmark provenance in attrs.
for df in datasets:
    name = df.attrs["dataset_name"]
    prov = df.attrs.get("provenance")
    prov0 = prov[0] if isinstance(prov, list) else (prov or {})
    missing = [c for c in target_columns if c not in df.columns]
    extra = [c for c in df.columns if c not in target_columns]
    print(f"{name}: {len(df)} rows, {len(df.columns)} cols, "
          f"missing={missing}, extra={extra} | "
          f"source={prov0.get('source')} ({prov0.get('naming_convention')})")

products_1: 812 rows, 27 cols, missing=[], extra=[] | source=None (None)
products_2: 812 rows, 27 cols, missing=[], extra=[] | source=None (None)
products_3: 762 rows, 27 cols, missing=[], extra=[] | source=None (None)
products_4: 626 rows, 27 cols, missing=[], extra=[] | source=None (None)


### Step 2: Data profiling

In [3]:
profiling_datasets = [
    df_final_1,
    df_final_2,
    df_final_3,
    df_final_4
]

names = ["products_1", "products_2", "products_3", "products_4"]


total_records = sum(len(df) for df in profiling_datasets)
print(f"Total records across all profiling_datasets: {total_records:,}")

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(profiling_datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

Total records across all profiling_datasets: 3,012
products_1:
  Rows: 812
  Columns: 27
  Total nulls: 9,142
  Null percentage: 41.7%
  Null counts per column:
    brand: 27 (3.3%)
    price: 40 (4.9%)
    priceCurrency: 43 (5.3%)
    model: 64 (7.9%)
    model_number: 478 (58.9%)
    chipset_name: 581 (71.6%)
    vram_gb: 583 (71.8%)
    storage_gb: 241 (29.7%)
    read_speed_mb_s: 661 (81.4%)
    write_speed_mb_s: 690 (85.0%)
    bus_type: 184 (22.7%)
    interface_type: 376 (46.3%)
    width_mm: 767 (94.5%)
    length_mm: 767 (94.5%)
    height_mm: 766 (94.3%)
    weight_g: 786 (96.8%)
    storage_connection_type: 399 (49.1%)
    memory_type: 602 (74.1%)
    color: 692 (85.2%)
    form_factor: 395 (48.6%)

products_2:
  Rows: 812
  Columns: 27
  Total nulls: 9,269
  Null percentage: 42.3%
  Null counts per column:
    brand: 41 (5.0%)
    price: 58 (7.1%)
    priceCurrency: 58 (7.1%)
    model: 70 (8.6%)
    model_number: 482 (59.4%)
    chipset_name: 581 (71.6%)
    vram_gb: 586 (

{'rows': 626,
 'columns': 27,
 'nulls_total': 7166,
 'nulls_per_column': {'id': 0,
  'brand': 34,
  'title': 0,
  'description': 0,
  'price': 43,
  'priceCurrency': 44,
  'cluster_id': 0,
  'url': 0,
  'title_description': 0,
  'model': 49,
  'model_number': 376,
  'product_type': 1,
  'chipset_name': 443,
  'vram_gb': 449,
  'storage_gb': 196,
  'read_speed_mb_s': 520,
  'write_speed_mb_s': 538,
  'bus_type': 139,
  'interface_type': 293,
  'width_mm': 599,
  'length_mm': 602,
  'height_mm': 591,
  'weight_g': 605,
  'storage_connection_type': 306,
  'memory_type': 464,
  'color': 574,
  'form_factor': 300},
 'dtypes': {'id': 'int64',
  'brand': 'object',
  'title': 'object',
  'description': 'object',
  'price': 'float64',
  'priceCurrency': 'object',
  'cluster_id': 'int64',
  'url': 'object',
  'title_description': 'object',
  'model': 'object',
  'model_number': 'object',
  'product_type': 'object',
  'chipset_name': 'object',
  'vram_gb': 'float64',
  'storage_gb': 'float64',
  

In [4]:
coverage = profiler.analyze_coverage(
    datasets=profiling_datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print(" Attribute coverage across profiling_datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

 Attribute coverage across profiling_datasets:


,attribute,products_1_count,products_1_pct,products_1_coverage,products_1_samples,products_2_count,products_2_pct,products_2_coverage,products_2_samples,products_3_count,products_3_pct,products_3_coverage,products_3_samples,products_4_count,products_4_pct,products_4_coverage,products_4_samples,avg_coverage,max_coverage,datasets_with_attribute
0,brand,785/812,96.7%,0.966749,"['Gigabyte', 'Western Digital', 'Corsair']",771/812,95.0%,0.949507,"['Gigabyte', 'Western Digital', 'Corsair']",721/762,94.6%,0.946194,"['Gigabyte', 'Western Digital', 'Corsair']",592/626,94.6%,0.945687,"['Gigabyte', 'Western Digital', 'Corsair']",0.952034,0.966749,4
1,bus_type,628/812,77.3%,0.773399,"['SATA', 'PCI Express x4', 'SATA']",598/812,73.6%,0.736453,"['SATA', 'PCI Express x4', 'SATA']",573/762,75.2%,0.751969,"['SATA', 'SATA', 'PCI Express x16']",487/626,77.8%,0.777955,"['SATA', 'SATA', 'PCI Express x16']",0.759944,0.777955,4
2,chipset_name,231/812,28.4%,0.284483,"['GeForce RTX 3080', 'GeForce GTX 1650', 'GeFo...",231/812,28.4%,0.284483,"['GeForce RTX 3080', 'GeForce GTX 1650', 'GeFo...",219/762,28.7%,0.287402,"['GeForce RTX 3080', 'GeForce GTX 1650', 'GeFo...",183/626,29.2%,0.292332,"['GeForce RTX 3080', 'GeForce GTX 1650', 'GeFo...",0.287175,0.292332,4
3,cluster_id,812/812,100.0%,1.000000,"[1002037, 1004942, 1007272]",812/812,100.0%,1.000000,"[1002037, 1004942, 1007272]",762/762,100.0%,1.000000,"[1002037, 1004942, 1007272]",626/626,100.0%,1.000000,"[1002037, 1004942, 1007272]",1.000000,1.000000,4
4,color,120/812,14.8%,0.147783,"['black and red', 'Green', 'Black']",112/812,13.8%,0.137931,"['Black', 'Silver', 'Cobalt Trim']",81/762,10.6%,0.106299,"['svart/blå', 'Black/White', 'black']",52/626,8.3%,0.083067,"['ROSE GOLD', 'Black', 'Alb']",0.118770,0.147783,4
5,description,812/812,100.0%,1.000000,"['CUDA Cores: 8704, Boost Clock: 1800MHz, GDDR...",812/812,100.0%,1.000000,['Gigabyte NVIDIA GeForce RTX 3080 GAMING OC 1...,762/762,100.0%,1.000000,"['To Avail the offer Click Here', 'Western Dig...",626/626,100.0%,1.000000,['Gigabyte Video Card GV-N3080GAMING OC-10GD G...,1.000000,1.000000,4
6,form_factor,417/812,51.4%,0.513547,"['3.5-inch', 'M.2 2280', '3.5-inch']",410/812,50.5%,0.504926,"['3.5-inch', 'M.2 2280', '3.5-inch']",374/762,49.1%,0.490814,"['3.5-inch', '3.5-inch', 'M.2 2280']",326/626,52.1%,0.520767,"['3.5-inch', 'M.2 2280', '3.5-inch']",0.507513,0.520767,4
7,height_mm,46/812,5.7%,0.056650,"[122.0, 6.1, 40.6]",48/812,5.9%,0.059113,"[127.0, 10.5, 11.4]",37/762,4.9%,0.048556,"[130.9, 20.8, 7.0]",35/626,5.6%,0.055911,"[7.0, 15.0, 119.3]",0.055058,0.059113,4
8,id,812/812,100.0%,1.000000,"[12198483, 78378158, 80641070]",812/812,100.0%,1.000000,"[19126355, 42841911, 46775597]",762/762,100.0%,1.000000,"[46320085, 91583813, 86850217]",626/626,100.0%,1.000000,"[66956099, 5078198, 77571226]",1.000000,1.000000,4
9,interface_type,436/812,53.7%,0.536946,"['SATA III', 'NVMe', 'SATA III']",424/812,52.2%,0.522167,"['SATA III', 'NVMe', 'SATA III']",381/762,50.0%,0.500000,"['SATA III', 'NVMe', 'SATA III']",333/626,53.2%,0.531949,"['SATA III', 'SATA III', 'NVMe']",0.522766,0.536946,4



 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['brand', 'bus_type', 'chipset_name', 'cluster_id', 'color', 'description', 'form_factor', 'height_mm', 'id', 'interface_type', 'length_mm', 'memory_type', 'model', 'model_number', 'price', 'priceCurrency', 'product_type', 'read_speed_mb_s', 'storage_connection_type', 'storage_gb', 'title', 'title_description', 'url', 'vram_gb', 'weight_g', 'width_mm', 'write_speed_mb_s']


## Part 2: Entity Matching

In [5]:
# Set up logging
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.WARNING, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

### Step 1: Blocking

In [6]:
out_dir_blocker =OUTPUT_DIR / "Blocking" / "standard_blocker_on_product_type"
out_dir_blocker.mkdir(parents=True, exist_ok=True)


# 3. Initialize the Star Schema Blockers
standard_blocker_p1_p2 = StandardBlocker(
    products_1_cleaned, products_2_cleaned,
    on=['product_type'],
    batch_size=1000,
    output_dir=out_dir_blocker / "p1_p2_blocking",
    id_column='id'
)

standard_blocker_p1_p3 = StandardBlocker(
    products_1_cleaned, products_3_cleaned,
    on=['product_type'],
    batch_size=1000,
    output_dir=out_dir_blocker / "p1_p3_blocking",
    id_column='id'
)

standard_blocker_p1_p4 = StandardBlocker(
    products_1_cleaned, products_4_cleaned,
    on=[ 'product_type'],
    batch_size=1000,
    output_dir=out_dir_blocker / "p1_p4_blocking",
    id_column='id'
)

### Step 2: Evaluate Blocking Against Ground Truth

In [7]:
# Evaluate the blocker against the held-out test pairs for each dataset pair
test_gt_p1_p2 = load_csv(
    OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod2_test.csv",
    name="test_p1_p2", header=0, names=['id1', 'id2', 'label', 'ishard'], add_index=False)

results_p1_p2 = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_p1_p2,
    test_pairs=test_gt_p1_p2,
    out_dir=OUTPUT_DIR / "Blocking" / "blocking_eval_prod1_prod2"
)

test_gt_p1_p3 = load_csv(
    OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod3_test.csv",
    name="test_p1_p3", header=0, names=['id1', 'id2', 'label', 'ishard'], add_index=False)

results_p1_p3 = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_p1_p3,
    test_pairs=test_gt_p1_p3,
    out_dir=OUTPUT_DIR / "Blocking" / "blocking_eval_prod1_prod3"
)

test_gt_p1_p4 = load_csv(
    OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod4_test.csv",
    name="test_p1_p4", header=0, names=['id1', 'id2', 'label', 'ishard'], add_index=False)

results_p1_p4 = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_p1_p4,
    test_pairs=test_gt_p1_p4,
    out_dir=OUTPUT_DIR / "Blocking" / "blocking_eval_prod1_prod4"
)

In [8]:
# Build a summary table from the per-pair evaluation result dicts
summary = pd.DataFrame(
    [
        {"dataset_pair": "p1_p2", **results_p1_p2},
        {"dataset_pair": "p1_p3", **results_p1_p3},
        {"dataset_pair": "p1_p4", **results_p1_p4},
    ]
)[["dataset_pair", "pair_completeness", "reduction_ratio"]]

print("Summary of Blocking Results:")
display(summary)

Summary of Blocking Results:


,dataset_pair,pair_completeness,reduction_ratio
0,p1_p2,1.0,0.720010
1,p1_p3,1.0,0.719931
2,p1_p4,1.0,0.716824


### Step 3: Entity Matching with Comparators

In [9]:
# Custom hardware normalization (similar to music/companies)
# This removes noise like "GB", "TB", and punctuation to focus on model codes
def normalize_hardware_text(s: str) -> str: 
    if s is None:
        return ""
    # Remove standard units and punctuation to isolate model numbers (e.g., 980, SN850)
    s = re.sub(r"(?i)\b(gb|tb|ssd|nvme|internal|hhd|sata)\b", "", s)
    return re.sub(r"[^\w\s]|_", "", s).lower().strip()


comparators = [
    # 1. Title (Tokens) - Handles word shuffling
    StringComparator(
        column='title', 
        similarity_function='sorensen_dice', #got better results than jaccard
        tokenization='word',
        preprocess=normalize_hardware_text
    ),
    
    # 2. Brand - Switched to Jaccard since they are normalized
    StringComparator(
        column='brand',
        similarity_function='jaccard',
        tokenization='word',
        preprocess=str.lower
    ),
    
    # 3. Product Type - Switched to Jaccard for strict category matching
    StringComparator(
        column='product_type',
        similarity_function='jaccard',
        tokenization='word',
        preprocess=str.lower
    ),
    
    # 4. Storage GB - Essential to distinguish 500GB from 1000GB variants
    NumericComparator(
        column='storage_gb',
        method='relative_difference',
        max_difference=0.1 
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

### Step 4: Rule Based Matcher

In [10]:
matcher = RuleBasedMatcher()

# New Weights distribution (must sum to 1.0):
# [Title_sorensen_dice, Brand_Jaccard, product_Type_Jaccard, Storage_Numeric]
# current_weights = [0.35, 0.30, 0.20, 0.15] #good balance, decent F1
# current_weights = [0.35, 0.30, 0.25, 0.10] #good for all, slightly better F1 than above (2nd best)
# current_weights = [0.30, 0.25, 0.30, 0.15] #slightly worse
current_weights = [0.30, 0.30, 0.30, 0.10] #best so far
# current_weights = [0.45, 0.15, 0.10, 0.30] #perfect precision but sinks in recall and f1
# current_weights = [0.35, 0.30, 0.30, 0.05] #worse than above
# current_weights = [0.50, 0.15, 0.30, 0.05] #massive drop in recall and F1, even if precision is perfect, so not good
# current_weights = [0.40, 0.20, 0.30, 0.10] #2nd best

current_threshold = 0.70 

# Evaluation for P1 to P2
correspondences_p1_p2 = matcher.match(
    df_left=products_1_cleaned,
    df_right=products_2_cleaned,
    candidates=standard_blocker_p1_p2, 
    comparators=comparators, 
    weights=current_weights,
    threshold=current_threshold,
    id_column='id'
)

# Evaluation for P1 to P3
correspondences_p1_p3 = matcher.match(
    df_left=products_1_cleaned,
    df_right=products_3_cleaned,
    candidates=standard_blocker_p1_p3,
    comparators=comparators,
    weights=current_weights,
    threshold=current_threshold,
    id_column='id'
)

# Evaluation for P1 to P4
correspondences_p1_p4 = matcher.match(
    df_left=products_1_cleaned,
    df_right=products_4_cleaned,
    candidates=standard_blocker_p1_p4,
    comparators=comparators,
    weights=current_weights,
    threshold=current_threshold,
    id_column='id'
)

KeyboardInterrupt: 

### Step 5: Evaluate Matching Against Ground Truth

In [ ]:
# Evaluate matching results against the test ground truth for each pair
results_p1_p2 = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p2,
    test_pairs=test_gt_p1_p2,
)

results_p1_p3 = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p3,
    test_pairs=test_gt_p1_p3,
)

results_p1_p4 = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p4,
    test_pairs=test_gt_p1_p4,
)

matching_summary = pd.DataFrame(
    [
        {"dataset_pair": "p1_p2", **results_p1_p2},
        {"dataset_pair": "p1_p3", **results_p1_p3},
        {"dataset_pair": "p1_p4", **results_p1_p4},
    ]
)[["dataset_pair", "precision", "recall", "f1", "accuracy"]]

print("Summary of Matching Results:")
display(matching_summary)

Summary of Matching Results:


,dataset_pair,precision,recall,f1,accuracy
0,p1_p2,0.641129,0.913793,0.753555,0.775378
1,p1_p3,0.681373,0.874214,0.765840,0.775726
2,p1_p4,0.831933,0.951923,0.887892,0.880383


In [ ]:
# This prevents the logger from crashing when it can't print a special character
logging.raiseExceptions = False

print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p2,
    out_dir=str(OUTPUT_DIR / "cluster_analysis"/ "p1_p2_cluster_distribution")
)

print(f"\n Cluster Size Distribution Results for p1_p2:")
display(cluster_distribution)

# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "p1_p2_cluster_distribution" / "p1_p2_detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_p1_p2,
    out_path=cluster_details_path
)


Analyzing cluster size distribution in our entity matching results...

 Cluster Size Distribution Results for p1_p2:


,cluster_size,frequency,percentage
0,2,13,23.636364
1,3,4,7.272727
2,4,5,9.090909
3,5,1,1.818182
4,6,3,5.454545
5,7,1,1.818182
6,8,3,5.454545
7,10,1,1.818182
8,12,2,3.636364
9,14,2,3.636364


In [ ]:
# This prevents the logger from crashing when it can't print a special character
logging.raiseExceptions = False

print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p3,
    out_dir=str(OUTPUT_DIR / "cluster_analysis"/ "p1_p3_cluster_distribution")
)

print(f"\n Cluster Size Distribution Results for p1_p3:")
display(cluster_distribution)
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "p1_p3_cluster_distribution" / "p1_p3_detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_p1_p3,
    out_path=cluster_details_path
)


Analyzing cluster size distribution in our entity matching results...

 Cluster Size Distribution Results for p1_p3:


,cluster_size,frequency,percentage
0,2,16,28.571429
1,3,3,5.357143
2,4,4,7.142857
3,5,2,3.571429
4,6,1,1.785714
5,7,2,3.571429
6,8,4,7.142857
7,10,2,3.571429
8,14,1,1.785714
9,15,2,3.571429


In [ ]:
# This prevents the logger from crashing when it can't print a special character
logging.raiseExceptions = False

print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p4,
    out_dir=str(OUTPUT_DIR / "cluster_analysis"/ "p1_p4_cluster_distribution")
)

print(f"\n Cluster Size Distribution Results for p1_p4:")
display(cluster_distribution)

# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "p1_p4_cluster_distribution" / "p1_p4_detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_p1_p4,
    out_path=cluster_details_path
)

Analyzing cluster size distribution in our entity matching results...

 Cluster Size Distribution Results for p1_p4:


,cluster_size,frequency,percentage
0,2,6,13.953488
1,3,2,4.651163
2,4,4,9.302326
3,5,1,2.325581
4,6,2,4.651163
5,7,3,6.976744
6,8,2,4.651163
7,12,2,4.651163
8,13,1,2.325581
9,14,1,2.325581


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

### Step 6: 1:1 Refinement (Greedy vs Maximum Bipartite Matching)

In [ ]:
# Compare 1:1 refinement strategies on each candidate pool and pick the best F1 per dataset
refiners = {
    "greedy": GreedyOneToOneMatchingAlgorithm(),
    "mbm": MaximumBipartiteMatching(),
}

pairs = {
    "p1_p2": (correspondences_p1_p2, test_gt_p1_p2),
    "p1_p3": (correspondences_p1_p3, test_gt_p1_p3),
    "p1_p4": (correspondences_p1_p4, test_gt_p1_p4),
}

candidate_correspondences = {}
rows = []
for pair_name, (corr, gt) in pairs.items():
    candidate_correspondences[pair_name] = {"baseline": corr}
    rows.append({"dataset_pair": pair_name, "method": "baseline", **EntityMatchingEvaluator.evaluate_matching(correspondences=corr, test_pairs=gt)})
    for method_name, clusterer in refiners.items():
        refined = clusterer.cluster(corr)
        candidate_correspondences[pair_name][method_name] = refined
        result = EntityMatchingEvaluator.evaluate_matching(
            correspondences=refined,
            test_pairs=gt,
            out_dir=OUTPUT_DIR / "debug_results_entity_matching" / f"{pair_name}_{method_name}",
        )
        rows.append({"dataset_pair": pair_name, "method": method_name, **result})

refinement_summary = pd.DataFrame(rows)[["dataset_pair", "method", "precision", "recall", "f1", "accuracy"]]
print("Refinement comparison (baseline vs Greedy vs Maximum Bipartite Matching):")
display(refinement_summary)

# Pick the correspondences with the best F1 per dataset pair (baseline is a valid winner)
best_per_pair = (
    refinement_summary.sort_values("f1", ascending=False)
    .drop_duplicates("dataset_pair")
    .set_index("dataset_pair")["method"]
    .to_dict()
)
print("Best method per dataset pair (by F1):", best_per_pair)

correspondences_p1_p2_refined = candidate_correspondences["p1_p2"][best_per_pair["p1_p2"]]
correspondences_p1_p3_refined = candidate_correspondences["p1_p3"][best_per_pair["p1_p3"]]
correspondences_p1_p4_refined = candidate_correspondences["p1_p4"][best_per_pair["p1_p4"]]

Refinement comparison (baseline vs Greedy vs Maximum Bipartite Matching):


,dataset_pair,method,precision,recall,f1,accuracy
0,p1_p2,baseline,0.641129,0.913793,0.753555,0.775378
1,p1_p2,greedy,0.933333,0.482759,0.636364,0.792657
2,p1_p2,mbm,0.607143,0.097701,0.168317,0.637149
3,p1_p3,baseline,0.681373,0.874214,0.765840,0.775726
4,p1_p3,greedy,0.978947,0.584906,0.732283,0.820580
5,p1_p3,mbm,0.882353,0.188679,0.310881,0.649077
6,p1_p4,baseline,0.831933,0.951923,0.887892,0.880383
7,p1_p4,greedy,0.984127,0.596154,0.742515,0.794258
8,p1_p4,mbm,1.000000,0.105769,0.191304,0.555024


Best method per dataset pair (by F1): {'p1_p4': 'baseline', 'p1_p3': 'baseline', 'p1_p2': 'baseline'}


### Step 7: ML-Based Matcher

In [ ]:
# Train and evaluate an ML matcher on each candidate pool, then 1:1-refine the result
datasets = {
    "p1_p2": (products_1_cleaned, products_2_cleaned, standard_blocker_p1_p2,
              "prod1_to_prod2_train.csv", test_gt_p1_p2),
    "p1_p3": (products_1_cleaned, products_3_cleaned, standard_blocker_p1_p3,
              "prod1_to_prod3_train.csv", test_gt_p1_p3),
    "p1_p4": (products_1_cleaned, products_4_cleaned, standard_blocker_p1_p4,
              "prod1_to_prod4_train.csv", test_gt_p1_p4),
}

extractor = FeatureExtractor(comparators)
ml_refiner = MaximumBipartiteMatching()
ml_results = {}
ml_correspondences = {}
for pair_name, (df_left, df_right, blocker, train_file, gt) in datasets.items():
    train_pairs = load_csv(
        OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / train_file,
        add_index=False,
    )
    train_features = extractor.create_features(
        df_left=df_left, df_right=df_right, pairs=train_pairs,
        id_column='id', labels=train_pairs['label'],
    )
    X_train = train_features.drop(['label', 'id1', 'id2'], axis=1)
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, train_features['label'])

    matcher = MLBasedMatcher(extractor)
    raw_ml = matcher.match(
        df_left=df_left, df_right=df_right, candidates=blocker,
        id_column='id', trained_classifier=clf,
        threshold=0.45, use_probabilities=True,
    )
    # Refine ML output to 1:1 so it can be fused without producing mega-clusters
    ml_correspondences[pair_name] = ml_refiner.cluster(raw_ml)
    ml_results[pair_name] = EntityMatchingEvaluator.evaluate_matching(
        correspondences=ml_correspondences[pair_name], test_pairs=gt,
    )

# Compare ML matcher against the best rule-based result per dataset (from refinement_summary)
best_rule_based = (
    refinement_summary.sort_values("f1", ascending=False)
    .drop_duplicates("dataset_pair")
    .set_index("dataset_pair")
)

comparison_rows = []
for pair_name, ml_res in ml_results.items():
    rule_row = best_rule_based.loc[pair_name]
    comparison_rows.append({
        "dataset_pair": pair_name,
        "rule_method": rule_row["method"],
        "rule_f1": rule_row["f1"],
        "ml_f1": ml_res["f1"],
        "f1_delta": ml_res["f1"] - rule_row["f1"],
        "rule_precision": rule_row["precision"],
        "ml_precision": ml_res["precision"],
        "rule_recall": rule_row["recall"],
        "ml_recall": ml_res["recall"],
    })

ml_vs_rule = pd.DataFrame(comparison_rows)
ml_vs_rule["winner"] = ml_vs_rule.apply(
    lambda row: "ml" if row["ml_f1"] > row["rule_f1"] else row["rule_method"], axis=1,
)
print("ML matcher (1:1 refined) vs best rule-based method per dataset:")
display(ml_vs_rule)

winners = ml_vs_rule.set_index("dataset_pair")["winner"].to_dict()
print("Final candidate pool per dataset pair:", winners)

fusion_safe_refiner = MaximumBipartiteMatching()
def pick(pair_name):
    method = winners[pair_name]
    chosen = ml_correspondences[pair_name] if method == "ml" else candidate_correspondences[pair_name][method]
    # baseline is raw N:M and would collapse fusion into mega-clusters; force 1:1 (no-op for already-refined pools)
    return fusion_safe_refiner.cluster(chosen) if method == "baseline" else chosen

correspondences_p1_p2_refined = pick("p1_p2")
correspondences_p1_p3_refined = pick("p1_p3")
correspondences_p1_p4_refined = pick("p1_p4")

print(
    "Refined link counts:",
    {
        "p1_p2": len(correspondences_p1_p2_refined),
        "p1_p3": len(correspondences_p1_p3_refined),
        "p1_p4": len(correspondences_p1_p4_refined),
    },
)

ML matcher (1:1 refined) vs best rule-based method per dataset:


,dataset_pair,rule_method,rule_f1,ml_f1,f1_delta,rule_precision,ml_precision,rule_recall,ml_recall,winner
0,p1_p2,baseline,0.753555,0.086486,-0.667068,0.641129,0.727273,0.913793,0.045977,baseline
1,p1_p3,baseline,0.765840,0.095238,-0.670602,0.681373,0.888889,0.874214,0.050314,baseline
2,p1_p4,baseline,0.887892,0.055556,-0.832337,0.831933,0.750000,0.951923,0.028846,baseline


Final candidate pool per dataset pair: {'p1_p2': 'baseline', 'p1_p3': 'baseline', 'p1_p4': 'baseline'}
Refined link counts: {'p1_p2': 745, 'p1_p3': 707, 'p1_p4': 580}


## Part 3: Data Fusion

### we fist split our data into test and validation 

In [ ]:
# Set the base directory to where the notebook is
BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "input" / "fusion"

# Load using the short, relative path
val_set = pd.read_csv(DATA_DIR / 'fusion_validation_set.csv')
test_set = pd.read_csv(DATA_DIR / 'fusion_test_set.csv')

print(f"Loaded Validation: {len(val_set)} rows")
print(f"Loaded Test: {len(test_set)} rows")

Loaded Validation: 100 rows
Loaded Test: 100 rows


#### we need to see which file is more trustworthy so we use or fusion validation set to check

In [ ]:
def hardware_strict_spec_match(fused_value, expected_value):
    """Compare numeric tokens strictly; prevents 'PCIE x8' matching 'PCIE x16'."""
    if pd.isna(fused_value) or pd.isna(expected_value):
        return False
    f = str(fused_value).lower()
    e = str(expected_value).lower()
    if re.findall(r'\d+', f) != re.findall(r'\d+', e):
        return False
    f_clean = re.sub(r'[^a-z0-9]', '', f)
    e_clean = re.sub(r'[^a-z0-9]', '', e)
    return e_clean in f_clean or f_clean in e_clean

audit_strategy = DataFusionStrategy('validation_audit')

for attr in ['brand', 'product_type', 'model_number']:
    audit_strategy.add_evaluation_function(attr, exact_match)

num_attrs = ['vram_gb', 'storage_gb', 'read_speed_mb_s', 'write_speed_mb_s',
             'width_mm', 'length_mm', 'height_mm', 'weight_g']
for attr in num_attrs:
    audit_strategy.add_evaluation_function(attr, numeric_tolerance_match, tolerance=0.15)

tech_attrs = ['bus_type', 'interface_type', 'chipset_name',
              'storage_connection_type', 'memory_type', 'form_factor']
for attr in tech_attrs:
    audit_strategy.add_evaluation_function(attr, hardware_strict_spec_match)

evaluator = DataFusionEvaluator(audit_strategy)

sources = [products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned]
names = ["p1", "p2", "p3", "p4"]

audit_summary = []
for df_source, name in zip(sources, names):
    id_col = 'id_left' if name == 'p1' else 'id_right'
    relevant_gt = val_set[(val_set['source_left'] == name) | (val_set['source_right'] == name)]
    results = evaluator.evaluate(
        fused_df=df_source,
        fused_id_column='id',
        gold_df=relevant_gt,
        gold_id_column=id_col,
    )
    audit_summary.append({
        'Source': name,
        'Overall_Accuracy': results.get('overall_accuracy'),
        'Details': results.get('per_attribute_scores'),
    })

report = pd.DataFrame(audit_summary).set_index('Source')
print("Validation audit results:")
display(report[['Overall_Accuracy']])

Validation audit results:


,Overall_Accuracy
Source,
p1,0.455550
p2,0.359003
p3,0.332036
p4,0.357834


#### based on trust scores. we use this order of trust

In [ ]:
# Set P1 ID as the master ID for the fused records
products_1_cleaned["p1_id"] = products_1_cleaned["id"]

# Trust scores (1 = lowest, 3 = highest)
products_1_cleaned.attrs["trust_score"] = 3
products_2_cleaned.attrs["trust_score"] = 2
products_3_cleaned.attrs["trust_score"] = 1
products_4_cleaned.attrs["trust_score"] = 2

all_correspondences = pd.concat([
    correspondences_p1_p2_refined,
    correspondences_p1_p3_refined,
    correspondences_p1_p4_refined,
], ignore_index=True)

# concat can upcast int id columns to float when frames differ in dtype; cast back
all_correspondences['id1'] = all_correspondences['id1'].astype('int64')
all_correspondences['id2'] = all_correspondences['id2'].astype('int64')

print(f"Total refined links for fusion: {len(all_correspondences):,}")

Total refined links for fusion: 2,032


### Step 1: Define Fusion Strategy

In [ ]:
strategy = DataFusionStrategy('hardware_fusion_strategy')

# 1. Identity
for attr in ['brand', 'product_type', 'model_number']:
    strategy.add_attribute_fuser(attr, voting)

# 2. Performance Specs (Numbers)
for attr in ['vram_gb', 'storage_gb', 'read_speed_mb_s', 'write_speed_mb_s']:
    strategy.add_attribute_fuser(attr, minimum) #i think min could be better. lets see

# 3. Technical & Dimensions (Using Trust)
tech_and_dim_attrs = [
    'chipset_name', 'bus_type', 'interface_type', 'storage_connection_type', 
    'memory_type', 'form_factor', 'width_mm', 'length_mm', 'height_mm', 'weight_g'
]
for attr in tech_and_dim_attrs:
    strategy.add_attribute_fuser(attr, prefer_higher_trust, trust_key="trust_score")

# strategy.add_attribute_fuser('title', longest_string)

### Step 2: Run Fusion

In [ ]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json', 
                          debug_file=OUTPUT_DIR / "data_fusion" / "hardware_fusion_debug.jsonl")

fused = engine.run(
    datasets=[products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False, # Set to True if you want to keep products that didn't have matches
)   

print(f"Final Fused Rows: {len(fused):,}")
display(fused.head(1)) #maybe lower to 1 because the ouput is huge

Final Fused Rows: 768


,_id,_fusion_sources,_fusion_source_datasets,brand,bus_type,chipset_name,cluster_id,color,description,form_factor,...,storage_gb,title,title_description,url,vram_gb,weight_g,width_mm,write_speed_mb_s,_fusion_confidence,_fusion_metadata
0,12198483,"[12198483, 61120002, 71212847, 9127388]","[products_1, products_2, products_3, products_4]",Gigabyte,PCI Express x16,GeForce RTX 3080,1002037,None,"CUDA Cores: 8704, Boost Clock: 1800MHz, GDDR6X...",None,...,NaN,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,https://www.novatech.co.uk/products/gigabyte-n...,4.0,NaN,NaN,NaN,0.407407,"{'_id_rule': 'first_non_null', '_id_inputs': [..."


### Step 3: Evaluate Data Fusion

In [ ]:
# 1. Identity (Must be exact)
strategy.add_evaluation_function("brand", exact_match)
strategy.add_evaluation_function("product_type", exact_match)

# 2. Performance & Capacity (Allow 15% tolerance for rounding)
for attr in ['vram_gb', 'storage_gb', 'read_speed_mb_s', 'write_speed_mb_s']:
    strategy.add_evaluation_function(attr, numeric_tolerance_match, tolerance=0.15)

# 3. Dimensions (Allow 15% tolerance)
for attr in ['width_mm', 'length_mm', 'height_mm', 'weight_g']:
    strategy.add_evaluation_function(attr, numeric_tolerance_match, tolerance=0.15)

# 4. Technical Strings (Using Strict Number Matcher)
# This prevents "PCIe x8" from matching "PCIe x16"
for attr in ['chipset_name', 'bus_type', 'interface_type', 'memory_type']:
    strategy.add_evaluation_function(attr, hardware_strict_spec_match)



# Filter for only the verified rows
val_set_filled = val_set[val_set['filled'] == 'y'].copy()
test_set_filled = test_set[test_set['filled'] == 'y'].copy()
#count verified rows
print(f"Verified rows in Validation Set: {len(val_set_filled)}")
print(f"Verified rows in Test Set: {len(test_set_filled)}")

# Ensure numeric columns are actually numeric
for col in ['vram_gb', 'storage_gb', 'read_speed_mb_s', 'write_speed_mb_s']:
    val_set_filled[col] = pd.to_numeric(val_set_filled[col], errors='coerce')



# Initialize the evaluator
evaluator = DataFusionEvaluator(
    strategy, 
    debug=True, 
    debug_file=OUTPUT_DIR / "data_fusion" / "hardware_eval_debug.jsonl", 
    debug_format="json"
)

# Run the evaluation
print("Evaluating hardware fusion results...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,           # Your fused Golden Records
    fused_id_column='p1_id',  # The master ID column you created
    gold_df=test_set_filled,   # val data. will run now with test set after making a few more adjustments. #val_set_filled
    gold_id_column='id_left'  # The P1 ID column in the validation set
)

# Display results
print("\nHardware Fusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")    

[WARNING] PyDI.fusion.evaluation - Missing 7 expected/reference records in fused dataset: 325545, 20960713, 9908573, 74156633, 35966001, ...


Verified rows in Validation Set: 100
Verified rows in Test Set: 100
Evaluating hardware fusion results...

Hardware Fusion Evaluation Results:
  overall_accuracy: 0.370
  macro_accuracy: 0.398
  num_evaluated_records: 93
  num_evaluated_attributes: 26
  total_evaluations: 2031
  total_correct: 751
  model_number_accuracy: 0.435
  model_number_count: 92
  url_accuracy: 0.172
  url_count: 93
  height_mm_accuracy: 0.088
  height_mm_count: 91
  interface_type_accuracy: 0.237
  interface_type_count: 93
  write_speed_mb_s_accuracy: 0.379
  write_speed_mb_s_count: 58
  memory_type_accuracy: 0.667
  memory_type_count: 30
  width_mm_accuracy: 0.088
  width_mm_count: 91
  model_accuracy: 0.283
  model_count: 92
  chipset_name_accuracy: 0.957
  chipset_name_count: 23
  description_accuracy: 0.172
  description_count: 93
  vram_gb_accuracy: 0.565
  vram_gb_count: 23
  brand_accuracy: 0.968
  brand_count: 93
  storage_connection_type_accuracy: 0.435
  storage_connection_type_count: 69
  priceCurren